# Kenya Gazette PDF to structured JSON (Docling + Spatial Reorder)

This notebook runs **Docling** on one or more Gazette PDFs and builds a **single JSON-friendly record per file** with:

- PDF metadata: inferred title, file name, size in bytes, page count
- Full **Docling** exports: plain text, Markdown, and a compact document summary (full `export_to_dict` is optional — it can be large)
- **Spatial reading-order correction**: reorders text elements using bounding-box coordinates to fix two-column reading order (left column top-to-bottom, then right column, then full-width zones)
- **Gazette notices** split by the phrase `GAZETTE NOTICE NO.` (Kenya-style), with notice number, header line, and full notice text

**Input:** drop PDFs into the `pdfs/` folder next to this notebook (or change `PDF_DIR` below). Use **Choose which PDFs** to run on every file or only a chosen subset.

> **See also:** `gazette_docling_pipeline.ipynb` for the original pipeline without spatial reordering.

## Environment

Use the project virtual environment (already created in this folder as `.venv`):

- **Windows (PowerShell):** `.\.venv\Scripts\Activate.ps1`
- **Then:** `python -m pip install -r requirements.txt` if needed
- **Jupyter kernel:** register with `python -m ipykernel install --user --name=gazette-docling --display-name="Python (gazette-docling)"` using the venv’s `python`.

Select that kernel in this notebook before running cells.

In [22]:
from __future__ import annotations

import json
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from docling.document_converter import DocumentConverter
from docling_core.types.doc.labels import DocItemLabel

# Project root = folder containing this notebook
PROJECT_ROOT = Path.cwd().resolve()
PDF_DIR = PROJECT_ROOT / "pdfs"
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PDF_DIR:", PDF_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

PDF_DIR: C:\Users\Ronald Wahome\Documents\gazette_test_inputs\pdfs
OUTPUT_DIR: C:\Users\Ronald Wahome\Documents\gazette_test_inputs\output


## Gazette notice splitting

Notices are detected by a case-insensitive match on **`GAZETTE NOTICE NO.`** followed by an optional notice identifier (digits, letters, slashes, hyphens). Each segment runs until the next such marker or end of document.

You can tune `NOTICE_PATTERN` if your issues use a different wording.

In [23]:
# Match the start of each notice; group 1 = notice number / reference (best effort)
# The \.? after NOTICE handles OCR artifacts like "GAZETTE NOTICE. NO."
NOTICE_PATTERN = re.compile(
    r"(?is)\bGAZETTE\s+NOTICE\.?\s+NO\.?\s*[:\-]?\s*([0-9A-Za-z\/\-]*)"
)


def split_gazette_notices(full_text: str) -> list[dict[str, Any]]:
    """Split flat text into notice blocks after 'GAZETTE NOTICE NO.'."""
    if not full_text or not full_text.strip():
        return []

    matches = list(NOTICE_PATTERN.finditer(full_text))
    if not matches:
        return [
            {
                "gazette_notice_no": None,
                "gazette_notice_header": None,
                "gazette_notice_full_text": full_text.strip(),
                "other_attributes": {
                    "reason": "no GAZETTE NOTICE NO. markers found; entire text returned as one block",
                },
            }
        ]

    notices: list[dict[str, Any]] = []
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(full_text)
        block = full_text[start:end].strip()
        raw_no = (m.group(1) or "").strip()
        first_line, _, rest = block.partition("\n")
        header_candidate = first_line.strip()
        if len(header_candidate) > 500:
            header_candidate = header_candidate[:500] + "…"

        notices.append(
            {
                "gazette_notice_no": raw_no or None,
                "gazette_notice_header": header_candidate,
                "gazette_notice_full_text": block,
                "other_attributes": {
                    "char_span_start": start,
                    "char_span_end": end,
                    "lines_after_header": rest.count("\n") + (1 if rest else 0),
                },
            }
        )
    return notices


def extract_title_from_docling(doc) -> str:
    for item in getattr(doc, "texts", []) or []:
        if getattr(item, "label", None) == DocItemLabel.TITLE and getattr(item, "text", None):
            return str(item.text).strip()
    return ""


def docling_export_summary(doc_dict: dict[str, Any]) -> dict[str, Any]:
    """Small fingerprint of the Docling JSON without dumping huge trees twice."""
    texts = doc_dict.get("texts") or []
    return {
        "schema_name": doc_dict.get("schema_name"),
        "version": doc_dict.get("version"),
        "name": doc_dict.get("name"),
        "texts_count": len(texts) if isinstance(texts, list) else None,
        "tables_count": len(doc_dict.get("tables") or []) if isinstance(doc_dict.get("tables"), list) else None,
        "pictures_count": len(doc_dict.get("pictures") or []) if isinstance(doc_dict.get("pictures"), list) else None,
        "pages_count": len(doc_dict.get("pages") or []) if isinstance(doc_dict.get("pages"), list) else None,
    }

## Spatial reading-order correction

Docling's internal reading-order predictor uses heuristics that can misorder text in two-column layouts — a notice that starts at the bottom of the left column and continues at the top of the right column may be split or interleaved with unrelated content.

The `reorder_by_spatial_position()` function fixes this by using each element's bounding-box coordinates from the Docling export dict:

1. **Group** all text (and table) elements by page number
2. **Detect columns** by checking whether element centers fall left or right of the page midpoint
3. **Find the column-zone boundary** — the y-coordinate where the shorter column ends
4. **Reclassify** elements below that boundary as full-width (even if left-aligned)
5. **Sort**: furniture (headers) → left column top-to-bottom → right column top-to-bottom → full-width zone top-to-bottom
6. **Concatenate** to produce corrected plain text

In [24]:
from dataclasses import dataclass as _dc


@_dc
class _BBoxElement:
    """Lightweight carrier for a text/table element with its spatial position."""
    page_no: int
    label: str
    content_layer: str
    text: str
    left: float
    top: float
    right: float
    bottom: float
    center_x: float
    self_ref: str

    @property
    def width(self) -> float:
        return self.right - self.left


def _table_to_text(data: dict[str, Any]) -> str:
    """Build a pipe-delimited text representation of a table from its grid data."""
    grid = data.get("grid")
    if not grid:
        cells = data.get("table_cells") or []
        return " | ".join(c.get("text", "") for c in cells if c.get("text"))
    rows: list[str] = []
    for row in grid:
        cols = [cell.get("text", "") for cell in row]
        rows.append(" | ".join(cols))
    return "\n".join(rows)


def _extract_elements(doc_dict: dict[str, Any]) -> list[_BBoxElement]:
    """Pull every text and table element that has provenance bbox data."""
    elements: list[_BBoxElement] = []

    for item in doc_dict.get("texts") or []:
        provs = item.get("prov") or []
        if not provs:
            continue
        prov = provs[0]
        bbox = prov.get("bbox")
        if not bbox:
            continue
        text = item.get("text") or item.get("orig") or ""
        if not text.strip():
            continue
        l, t, r, b = bbox["l"], bbox["t"], bbox["r"], bbox["b"]
        elements.append(_BBoxElement(
            page_no=prov["page_no"],
            label=item.get("label", "text"),
            content_layer=item.get("content_layer", "body"),
            text=text,
            left=l, top=t, right=r, bottom=b,
            center_x=(l + r) / 2,
            self_ref=item.get("self_ref", ""),
        ))

    for item in doc_dict.get("tables") or []:
        provs = item.get("prov") or []
        if not provs:
            continue
        prov = provs[0]
        bbox = prov.get("bbox")
        if not bbox:
            continue
        text = _table_to_text(item.get("data") or {})
        if not text.strip():
            continue
        l, t, r, b = bbox["l"], bbox["t"], bbox["r"], bbox["b"]
        elements.append(_BBoxElement(
            page_no=prov["page_no"],
            label="table",
            content_layer=item.get("content_layer", "body"),
            text=text,
            left=l, top=t, right=r, bottom=b,
            center_x=(l + r) / 2,
            self_ref=item.get("self_ref", ""),
        ))

    return elements


def _get_page_dimensions(doc_dict: dict[str, Any]) -> dict[int, tuple[float, float]]:
    """Return {page_no: (width, height)} from the pages map."""
    dims: dict[int, tuple[float, float]] = {}
    pages = doc_dict.get("pages") or {}
    for _key, pinfo in pages.items():
        sz = pinfo.get("size") or {}
        pno = pinfo.get("page_no")
        if pno is not None and sz:
            dims[pno] = (sz.get("width", 595.0), sz.get("height", 842.0))
    return dims


# Minimum fraction of page text-area width for an element to count as full-width
_FULLWIDTH_RATIO = 0.55
# How far above the first centered/full-width anchor to extend the zone boundary
_FW_TRANSITION_TOLERANCE = 50.0


def _reorder_page(
    page_elements: list[_BBoxElement],
    page_width: float,
) -> list[_BBoxElement]:
    """Reorder elements on a single page: left col -> right col -> full-width bottom."""

    mid_x = page_width / 2.0
    text_area_width = page_width - 100

    furniture: list[_BBoxElement] = []
    left_candidates: list[_BBoxElement] = []
    right_candidates: list[_BBoxElement] = []
    full_width: list[_BBoxElement] = []

    for el in page_elements:
        if el.content_layer == "furniture":
            furniture.append(el)
            continue

        clearly_right = el.left >= mid_x
        clearly_left = el.right <= mid_x
        spans_both = el.left < mid_x and el.right > mid_x
        wide_enough = el.width > text_area_width * _FULLWIDTH_RATIO
        centered = (spans_both
                    and not wide_enough
                    and abs(el.center_x - mid_x) < 80
                    and el.width < text_area_width * 0.45)

        if spans_both and wide_enough:
            full_width.append(el)
        elif centered:
            full_width.append(el)
        elif clearly_right:
            right_candidates.append(el)
        elif clearly_left:
            left_candidates.append(el)
        elif el.center_x < mid_x:
            left_candidates.append(el)
        else:
            right_candidates.append(el)

    # --- Single-column shortcut ---
    # If only one side of the page has candidates the page is not
    # two-column.  Merge everything and sort top-to-bottom.
    if not left_candidates or not right_candidates:
        all_body = left_candidates + right_candidates + full_width
        all_body.sort(key=lambda el: -el.top)
        return furniture + all_body

    # --- Detect the full-width zone transition ---
    # Use centered / spanning elements as anchors: the topmost such body
    # element marks where the full-width zone begins.  Everything at or
    # below that level (with tolerance) is reclassified out of the columns.
    fw_body = [el for el in full_width if el.content_layer != "furniture"]
    fw_transition_y = None

    if fw_body:
        fw_transition_y = max(el.top for el in fw_body)

        # Guard: only apply if the anchor is in the lower portion of the
        # page (below the median of column elements).  If centered content
        # sits above most column text it is likely a page-level heading,
        # not a bottom full-width zone.
        col_tops = [el.top for el in left_candidates + right_candidates]
        if col_tops:
            col_tops_sorted = sorted(col_tops, reverse=True)
            median_top = col_tops_sorted[len(col_tops_sorted) // 2]
            if fw_transition_y >= median_top:
                fw_transition_y = None

    if fw_transition_y is not None:
        threshold = fw_transition_y + _FW_TRANSITION_TOLERANCE

        left_col: list[_BBoxElement] = []
        for el in left_candidates:
            if el.top <= threshold:
                full_width.append(el)
            else:
                left_col.append(el)

        right_col: list[_BBoxElement] = []
        for el in right_candidates:
            if el.top <= threshold:
                full_width.append(el)
            else:
                right_col.append(el)
    else:
        # No full-width anchors (or anchors are at page top): fall back to
        # the "shorter column" heuristic.
        left_col = list(left_candidates)
        right_col = list(right_candidates)

        if left_col and right_col:
            left_bottom = min(el.bottom for el in left_col)
            right_bottom = min(el.bottom for el in right_col)
            col_zone_bottom = max(left_bottom, right_bottom)

            left_col = [el for el in left_col if el.top >= col_zone_bottom]
            full_width.extend(
                el for el in left_candidates if el.top < col_zone_bottom
            )
            right_col = [el for el in right_col if el.top >= col_zone_bottom]
            full_width.extend(
                el for el in right_candidates if el.top < col_zone_bottom
            )

    # Sort each group top-to-bottom (BOTTOMLEFT origin: higher t = higher on page)
    key_y = lambda el: -el.top
    furniture.sort(key=key_y)
    left_col.sort(key=key_y)
    right_col.sort(key=key_y)
    full_width.sort(key=key_y)

    return furniture + left_col + right_col + full_width


def reorder_by_spatial_position(doc_dict: dict[str, Any]) -> str:
    """Reorder all text elements by spatial position and return corrected plain text.

    For two-column pages the reading order becomes:
      page header → left column (top-to-bottom) → right column (top-to-bottom)
      → full-width zone at page bottom (top-to-bottom)
    Single-column pages are simply sorted top-to-bottom.
    """
    elements = _extract_elements(doc_dict)
    dims = _get_page_dimensions(doc_dict)

    # Group by page
    by_page: dict[int, list[_BBoxElement]] = {}
    for el in elements:
        by_page.setdefault(el.page_no, []).append(el)

    page_texts: list[str] = []
    for page_no in sorted(by_page):
        pw, _ph = dims.get(page_no, (595.0, 842.0))
        ordered = _reorder_page(by_page[page_no], pw)
        page_texts.append(
            "\n\n".join(el.text for el in ordered if el.content_layer != "furniture")
        )

    return "\n\n".join(page_texts)


print("Spatial reorder function ready.")

Spatial reorder function ready.


## Convert pipeline (with spatial reorder)

`DocumentConverter` is created once and reused. For each PDF we populate the output record and optionally write the full Docling JSON to `output/<stem>_docling.json`.

Gazette notices are now split from the **spatially reordered** plain text instead of the raw `export_to_text()` output. Both versions are kept in the record for comparison.

In [25]:
@dataclass
class GazettePipeline:
    converter: DocumentConverter = field(default_factory=DocumentConverter)
    include_full_docling_dict: bool = False

    def process_pdf(self, pdf_path: Path) -> dict[str, Any]:
        pdf_path = Path(pdf_path).resolve()
        stat = pdf_path.stat()

        result = self.converter.convert(str(pdf_path))
        doc = result.document

        title_guess = extract_title_from_docling(doc) or pdf_path.stem.replace("_", " ")
        page_count = len(doc.pages) if getattr(doc, "pages", None) else None

        plain = doc.export_to_text()
        md = doc.export_to_markdown()

        doc_dict = doc.export_to_dict()

        # Spatial reorder: reconstruct plain text using bbox coordinates
        # so two-column pages read left-col then right-col then full-width.
        plain_spatial = reorder_by_spatial_position(doc_dict)

        record: dict[str, Any] = {
            "pdf_title": title_guess,
            "pdf_file_name": pdf_path.name,
            "pdf_path": str(pdf_path),
            "pdf_size_bytes": stat.st_size,
            "pages": page_count,
            "docling": {
                "export_summary": docling_export_summary(doc_dict),
                "full_markdown": md,
                "full_plain_text": plain,
                "full_plain_text_spatial": plain_spatial,
            },
            "gazette_notices": split_gazette_notices(plain_spatial),
        }

        if self.include_full_docling_dict:
            record["docling"]["full_docling_document_dict"] = doc_dict

        # Create subdirectory for this PDF's outputs
        pdf_output_dir = OUTPUT_DIR / pdf_path.stem
        pdf_output_dir.mkdir(parents=True, exist_ok=True)

        out_json = pdf_output_dir / f"{pdf_path.stem}_gazette_spatial.json"
        with open(out_json, "w", encoding="utf-8") as f:
            json.dump(record, f, ensure_ascii=False, indent=2)

        out_spatial_txt = pdf_output_dir / f"{pdf_path.stem}_spatial.txt"
        with open(out_spatial_txt, "w", encoding="utf-8") as f:
            f.write(plain_spatial)

        out_markdown_default = pdf_output_dir / f"{pdf_path.stem}_docling_markdown.md"
        with open(out_markdown_default, "w", encoding="utf-8") as f:
            f.write(md)

        out_markdown_spatial = pdf_output_dir / f"{pdf_path.stem}_spatial_markdown.md"
        with open(out_markdown_spatial, "w", encoding="utf-8") as f:
            f.write(plain_spatial)

        docling_only = pdf_output_dir / f"{pdf_path.stem}_docling.json"
        with open(docling_only, "w", encoding="utf-8") as f:
            json.dump(doc_dict, f, ensure_ascii=False, indent=2)

        print(f"Wrote: {out_json}")
        print(f"Wrote: {out_spatial_txt}")
        print(f"Wrote: {out_markdown_default}")
        print(f"Wrote: {out_markdown_spatial}")
        print(f"Wrote: {docling_only}")
        return record


def run_pdfs(pipeline: GazettePipeline, pdf_paths: list[Path]) -> list[dict[str, Any]]:
    """Run the pipeline on an explicit ordered list of PDF paths."""
    if not pdf_paths:
        print("No PDF paths to process.")
        return []
    results: list[dict[str, Any]] = []
    for p in pdf_paths:
        print("\n--- Processing:", p.name, "---")
        results.append(pipeline.process_pdf(p))
    return results


def run_folder(pipeline: GazettePipeline, folder: Path | None = None) -> list[dict[str, Any]]:
    """Process every *.pdf in folder (same as selection mode \"all\")."""
    folder = Path(folder or PDF_DIR)
    pdfs = sorted(folder.glob("*.pdf"))
    if not pdfs:
        print(f"No PDF files in {folder}. Add .pdf files and re-run.")
        return []
    return run_pdfs(pipeline, pdfs)


def resolve_pdf_selection(
    mode: str,
    selected_names: list[str],
    folder: Path | None = None,
) -> list[Path]:
    """Resolve \"all\" or \"selected\" to a list of existing PDF paths under folder."""
    folder = Path(folder or PDF_DIR)
    pdfs = sorted(folder.glob("*.pdf"))
    if not pdfs:
        print(f"No PDF files in {folder}. Add .pdf files and re-run.")
        return []
    m = (mode or "all").strip().lower()
    if m == "all":
        return pdfs
    if m == "selected":
        if not selected_names:
            print("PDF_SELECTION_MODE is 'selected' but SELECTED_PDF_NAMES is empty. Nothing to do.")
            return []
        by_name = {p.name: p for p in pdfs}
        out: list[Path] = []
        missing: list[str] = []
        for raw in selected_names:
            name = raw.strip()
            if not name:
                continue
            p = by_name.get(name)
            if p is None:
                p = by_name.get(Path(name).name)
            if p is None:
                missing.append(name)
            elif p not in out:
                out.append(p)
        if missing:
            print("Warning: not found in", folder, ":", missing)
            print("Available PDFs:", [p.name for p in pdfs])
        return out
    raise ValueError('PDF_SELECTION_MODE must be "all" or "selected".')

## Choose which PDFs to process

Set **`PDF_SELECTION_MODE`** in the next cell:

- **`"all"`** — every `*.pdf` under `PDF_DIR`
- **`"selected"`** — only files listed in **`SELECTED_PDF_NAMES`** (exact file names as they appear in `pdfs/`)

Then run the pipeline cell.

## Run

First conversion may **download models** and take longer. Subsequent runs are faster.

In [26]:
# --- Configure which PDFs to process ---
# When mode is "selected", list exact file names as they appear in PDF_DIR (order is preserved).
# When mode is "all", SELECTED_PDF_NAMES is ignored.
PDF_SELECTION_MODE = "selected"  # "all" | "selected"
SELECTED_PDF_NAMES: list[str] = [
    "Kenya Gazette Vol CXINo 110.pdf",
]

pipeline = GazettePipeline(include_full_docling_dict=False)
pdf_paths = resolve_pdf_selection(PDF_SELECTION_MODE, SELECTED_PDF_NAMES)
print(
    "Mode:",
    PDF_SELECTION_MODE,
    "|",
    len(pdf_paths),
    "file(s):",
    [p.name for p in pdf_paths],
)
results = run_pdfs(pipeline, pdf_paths)

if results:
    sample = results[0]
    print("Keys:", list(sample.keys()))
    print("Pages:", sample.get("pages"))
    print("Notice count:", len(sample.get("gazette_notices") or []))
    display_snippet = json.dumps(sample, ensure_ascii=False, indent=2)
    if len(display_snippet) > 8000:
        print(display_snippet[:8000] + "\n... [truncated for display; see output JSON files] ...")
    else:
        print(display_snippet)

[INFO] 2026-04-11 23:41:55,044 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-11 23:41:55,045 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-11 23:41:55,074 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\Ronald Wahome\Documents\gazette_test_inputs\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-11 23:41:55,075 [RapidOCR] main.py:50: Using C:\Users\Ronald Wahome\Documents\gazette_test_inputs\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.pth


Mode: selected | 1 file(s): ['Kenya Gazette Vol CXINo 110.pdf']

--- Processing: Kenya Gazette Vol CXINo 110.pdf ---


[INFO] 2026-04-11 23:41:55,246 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-11 23:41:55,248 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-11 23:41:55,250 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\Ronald Wahome\Documents\gazette_test_inputs\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-04-11 23:41:55,251 [RapidOCR] main.py:50: Using C:\Users\Ronald Wahome\Documents\gazette_test_inputs\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-04-11 23:41:55,319 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-11 23:41:55,320 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-11 23:41:55,392 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\Ronald Wahome\Documents\gazette_test_inputs\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_rec_mobile.pth
[INFO] 2026-04-11 23:41:55,393 [RapidOCR] main.py:50: Using C:\Use

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Wrote: C:\Users\Ronald Wahome\Documents\gazette_test_inputs\output\Kenya Gazette Vol CXINo 110\Kenya Gazette Vol CXINo 110_gazette_spatial.json
Wrote: C:\Users\Ronald Wahome\Documents\gazette_test_inputs\output\Kenya Gazette Vol CXINo 110\Kenya Gazette Vol CXINo 110_spatial.txt
Wrote: C:\Users\Ronald Wahome\Documents\gazette_test_inputs\output\Kenya Gazette Vol CXINo 110\Kenya Gazette Vol CXINo 110_docling_markdown.md
Wrote: C:\Users\Ronald Wahome\Documents\gazette_test_inputs\output\Kenya Gazette Vol CXINo 110\Kenya Gazette Vol CXINo 110_spatial_markdown.md
Wrote: C:\Users\Ronald Wahome\Documents\gazette_test_inputs\output\Kenya Gazette Vol CXINo 110\Kenya Gazette Vol CXINo 110_docling.json
Keys: ['pdf_title', 'pdf_file_name', 'pdf_path', 'pdf_size_bytes', 'pages', 'docling', 'gazette_notices']
Pages: 52
Notice count: 184
{
  "pdf_title": "Kenya Gazette Vol CXINo 110",
  "pdf_file_name": "Kenya Gazette Vol CXINo 110.pdf",
  "pdf_path": "C:\\Users\\Ronald Wahome\\Documents\\gazette_tes

## Optional: embed full Docling tree in the main record

Set `include_full_docling_dict=True` if you need the complete `DoclingDocument` as JSON inside `gazette.json` (large).

In [27]:
# Example: toggle full dict for a single file
# p = PDF_DIR / "your_issue.pdf"
# if p.exists():
#     full_pipeline = GazettePipeline(include_full_docling_dict=True)
#     full_pipeline.process_pdf(p)

## Quick test: spatial reorder on existing Docling JSON

Run this cell to test the spatial reorder function against a previously exported Docling JSON file — no need to re-run the full PDF conversion. It prints the reordered text around notice 14096 so you can verify the column-crossing fix.

In [28]:
# # Quick test: load existing docling JSON and run spatial reorder
# _docling_json_path = OUTPUT_DIR / "Kenya Gazette Vol CXINo 110_docling.json"

# if _docling_json_path.exists():
#     with open(_docling_json_path, "r", encoding="utf-8") as _f:
#         _doc_dict = json.load(_f)

#     _spatial_text = reorder_by_spatial_position(_doc_dict)

#     # Show the region around notice 14096 to verify correct ordering
#     _marker = "GAZETTE NOTICE NO. 14096"
#     _idx = _spatial_text.find(_marker)
#     if _idx >= 0:
#         _start = max(0, _idx - 200)
#         _end = min(len(_spatial_text), _idx + 2000)
#         print(f"=== Spatial text around notice 14096 (chars {_start}–{_end}) ===\n")
#         print(_spatial_text[_start:_end])
#         print("\n=== End snippet ===")
#     else:
#         print("Notice 14096 not found in spatial text — check the reorder logic.")

#     # Also show notice 14098 to verify it now leads the PROBATE section
#     _marker2 = "GAZETTE NOTICE NO. 14098"
#     _idx2 = _spatial_text.find(_marker2)
#     if _idx2 >= 0:
#         _end2 = min(len(_spatial_text), _idx2 + 800)
#         print(f"\n=== Spatial text around notice 14098 (chars {_idx2}–{_end2}) ===\n")
#         print(_spatial_text[_idx2:_end2])
#         print("\n=== End snippet ===")

#     # Compare notice counts
#     _original_text_path = OUTPUT_DIR / "Kenya Gazette Vol CXINo 110.txt"
#     if _original_text_path.exists():
#         _orig = _original_text_path.read_text(encoding="utf-8")
#         _orig_notices = split_gazette_notices(_orig)
#         _spatial_notices = split_gazette_notices(_spatial_text)
#         print(f"\nOriginal notice count: {len(_orig_notices)}")
#         print(f"Spatial notice count:  {len(_spatial_notices)}")

#         # Show notice numbers side by side for the 14094-14098 range
#         print("\n--- Original order (14094-14099 range) ---")
#         for n in _orig_notices:
#             no = n.get("gazette_notice_no", "")
#             if no and 14094 <= int(no) <= 14099:
#                 print(f"  {n['gazette_notice_header']}  ({len(n['gazette_notice_full_text'])} chars)")

#         print("\n--- Spatial order (14094-14099 range) ---")
#         for n in _spatial_notices:
#             no = n.get("gazette_notice_no", "")
#             if no and 14094 <= int(no) <= 14099:
#                 print(f"  {n['gazette_notice_header']}  ({len(n['gazette_notice_full_text'])} chars)")
# else:
#     print(f"Docling JSON not found at {_docling_json_path}. Run the pipeline first.")